Chargement et découpage temporel

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

base = '/content/drive/MyDrive/EnergyMind_TSYP14/data/'
df = pd.read_csv(base + 'quartier_avec_anomalies.csv', parse_dates=['tstp'])
df = df.sort_values(['LCLid', 'tstp']).reset_index(drop=True)

DATE_SPLIT = pd.Timestamp('2013-11-28')
df['split'] = np.where(df['tstp'] < DATE_SPLIT, 'train', 'test')

print(df['split'].value_counts())
print(df.groupby('split')['is_data_quality_issue'].mean())  # vérifier que le test contient bien des anomalies

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
split
train    771456
test     181097
Name: count, dtype: int64
split
test     0.051271
train    0.056087
Name: is_data_quality_issue, dtype: float64


Baseline personnelle recalculée strictement sur le train

In [ ]:
df_train_ref = df[(df['split'] == 'train') & (df['anomaly_type'] == 'none')].copy()

baseline_train = df_train_ref.groupby(['LCLid', 'heure', 'jour_semaine'])['energy(kWh/hh)'].agg(
    mediane_train='median',
    mad_train=lambda x: (x - x.median()).abs().median(),
    n_train='count'
).reset_index()

# Baseline de secours, moins granulaire, pour les combinaisons avec trop peu d'observations
baseline_secours = df_train_ref.groupby(['LCLid', 'heure'])['energy(kWh/hh)'].agg(
    mediane_secours='median',
    mad_secours=lambda x: (x - x.median()).abs().median()
).reset_index()

baseline_train = baseline_train.merge(baseline_secours, on=['LCLid', 'heure'], how='left')
baseline_train['mediane_train'] = np.where(
    baseline_train['n_train'] < 15, baseline_train['mediane_secours'], baseline_train['mediane_train'])
baseline_train['mad_train'] = np.where(
    baseline_train['n_train'] < 15, baseline_train['mad_secours'], baseline_train['mad_train'])

df = df.drop(columns=['mediane_train', 'mad_train'], errors='ignore').merge(
    baseline_train[['LCLid', 'heure', 'jour_semaine', 'mediane_train', 'mad_train']],
    on=['LCLid', 'heure', 'jour_semaine'], how='left')

df['z_score_propre'] = (df['energy(kWh/hh)'] - df['mediane_train']) / (1.4826 * df['mad_train'].replace(0, np.nan))
df['z_score_propre'] = df['z_score_propre'].clip(-15, 15)

print(df['z_score_propre'].describe())
print("Combinaisons ayant nécessité la baseline de secours :", (baseline_train['n_train'] < 15).sum(), "/", len(baseline_train))

count    951249.000000
mean          0.774949
std           2.785770
min         -15.000000
25%          -0.591900
50%           0.025114
75%           0.959555
max          15.000000
Name: z_score_propre, dtype: float64
Combinaisons ayant nécessité la baseline de secours : 0 / 6888


Features ML : statistiques glissantes, lags, vitesse de variation

In [ ]:
def construire_features_ml(df):
    df = df.sort_values(['LCLid', 'tstp']).copy()
    g = df.groupby('LCLid')['energy(kWh/hh)']

    for fenetre, nom in [(6, '3h'), (48, '1j'), (336, '7j')]:
        df[f'roll_mean_{nom}'] = g.transform(lambda x: x.rolling(fenetre, min_periods=1).mean())
        df[f'roll_std_{nom}'] = g.transform(lambda x: x.rolling(fenetre, min_periods=1).std())

    for decalage, nom in [(1, '30min'), (48, '1j'), (336, '7j')]:
        df[f'lag_{nom}'] = g.transform(lambda x: x.shift(decalage))

    df['diff_1'] = g.transform(lambda x: x.diff())
    df['diff_2'] = g.transform(lambda x: x.diff().diff())

    return df

df = construire_features_ml(df)
print(df[['energy(kWh/hh)', 'roll_mean_1j', 'lag_1j', 'diff_1']].describe())

       energy(kWh/hh)   roll_mean_1j         lag_1j        diff_1
count   951249.000000  952230.000000  949329.000000  9.511850e+05
mean         0.448749       0.448434       0.448726  8.473432e-07
std          0.548991       0.409312       0.548812  3.085405e-01
min         -4.978000       0.010636      -4.978000 -2.538731e+01
25%          0.132000       0.222958       0.132000 -4.800000e-02
50%          0.254000       0.327417       0.254000 -1.000000e-03
75%          0.543000       0.536166       0.543000  4.300000e-02
max         28.085314       5.816646      28.085314  2.450931e+01


Cohérence physique, longueur de plage figée, contexte météo/calendaire

In [ ]:
df['S_calculee'] = (df['Voltage'] * df['Global_intensity']) / 1000
df['S_theorique'] = df['power_kW']  # approximation (pas de reactive_power disponible dans ce dataset)
df['ecart_physique'] = (df['S_theorique'] - df['S_calculee']).abs() / df['S_calculee'].replace(0, np.nan)

# Rendre la feature relative au niveau habituel de CE foyer (baseline train/normal uniquement)
ecart_baseline = df[(df['split'] == 'train') & (df['anomaly_type'] == 'none')].groupby('LCLid')['ecart_physique'].median()
df['ecart_physique_baseline'] = df['LCLid'].map(ecart_baseline)
df['ecart_physique_relatif'] = df['ecart_physique'] - df['ecart_physique_baseline']

def longueur_plage_figee(x):
    est_egal_precedent = (x.diff() == 0)
    groupes = (est_egal_precedent != est_egal_precedent.shift()).cumsum()
    return est_egal_precedent.groupby(groupes).cumsum()

df['longueur_plage_figee'] = df.groupby('LCLid')['energy(kWh/hh)'].transform(longueur_plage_figee)

df['vague_extreme'] = ((df['Temperature_synth'] > 32) | (df['Temperature_synth'] < 5)).astype(int)
df['est_weekend'] = (df['jour_semaine'] >= 5).astype(int)

print(df[['ecart_physique_relatif', 'longueur_plage_figee', 'vague_extreme']].describe())

       ecart_physique_relatif  longueur_plage_figee  vague_extreme
count           951249.000000         952553.000000  952553.000000
mean                 0.093775              0.220536       0.013603
std                  0.575776              2.080299       0.115838
min                 -0.866331              0.000000       0.000000
25%                 -0.113964              0.000000       0.000000
50%                  0.000319              0.000000       0.000000
75%                  0.174404              0.000000       0.000000
max                 64.121446             49.000000       1.000000


In [ ]:
df['is_anomalous_event'] = df['anomaly_type'] != 'none'
df['is_data_quality_issue'] = df['anomaly_type'].isin(
    ['equipment_fault', 'degraded_sensor', 'biased_reading', 'corrupted_falsified', 'natural_failure'])
df['is_genuine_event'] = df['anomaly_type'] == 'genuine_event'

print(df[['is_anomalous_event', 'is_data_quality_issue', 'is_genuine_event']].mean())

is_anomalous_event       0.058306
is_data_quality_issue    0.054984
is_genuine_event         0.003323
dtype: float64


Assemblage table ML finale, retrait des premières lignes sans historique

In [ ]:
colonnes_features = [
    'LCLid', 'tstp', 'split', 'energy(kWh/hh)', 'Voltage', 'Global_intensity',
    'Temperature_synth', 'Humidity_synth', 'heure', 'jour_semaine', 'est_weekend',
    'roll_mean_3h', 'roll_std_3h', 'roll_mean_1j', 'roll_std_1j', 'roll_mean_7j', 'roll_std_7j',
    'lag_30min', 'lag_1j', 'lag_7j', 'diff_1', 'diff_2',
    'z_score_propre', 'ecart_physique_relatif', 'longueur_plage_figee', 'vague_extreme',
    'fraction_foyers_extremes', 'etait_manquant',
    'anomaly_type', 'is_anomalous_event', 'is_data_quality_issue', 'is_genuine_event'
]

df_features = df[colonnes_features].copy()
df_features = df_features.dropna(subset=['lag_7j'])

print(df_features.shape)
print(df_features.groupby('split')[['is_anomalous_event', 'is_data_quality_issue', 'is_genuine_event']].mean())

df_features.to_csv(base + 'features_ml.csv', index=False)

(937761, 32)
       is_anomalous_event  is_data_quality_issue  is_genuine_event
split                                                             
test             0.053699               0.051178          0.002521
train            0.059398               0.055818          0.003579


Construction des tenseurs DL (fenêtrage 48 pas, stride 24)

In [ ]:
CANAUX = ['energy(kWh/hh)', 'Voltage', 'Global_intensity', 'Temperature_synth', 'Humidity_synth']
LONGUEUR_FENETRE = 48
STRIDE = 24

def construire_fenetres(df_foyer, longueur, stride):
    valeurs = df_foyer[CANAUX].to_numpy()
    masque_valide = df_foyer[CANAUX].notna().all(axis=1).to_numpy()
    labels_dq = df_foyer['is_data_quality_issue'].to_numpy()
    labels_genuine = df_foyer['is_genuine_event'].to_numpy()
    splits = df_foyer['split'].to_numpy()

    fenetres, masques = [], []
    y_dq_resume, y_dq_detail = [], []
    y_genuine_resume = []
    split_fenetre = []

    for depart in range(0, len(df_foyer) - longueur + 1, stride):
        fin = depart + longueur
        if not masque_valide[depart:fin].all():
            continue
        fenetres.append(valeurs[depart:fin])
        masques.append(masque_valide[depart:fin])
        y_dq_resume.append(labels_dq[depart:fin].any())
        y_dq_detail.append(labels_dq[depart:fin])
        y_genuine_resume.append(labels_genuine[depart:fin].any())

        s = splits[depart:fin]
        split_fenetre.append('train' if (s == 'train').all() else ('test' if (s == 'test').all() else 'mixte'))

    return fenetres, masques, y_dq_resume, y_dq_detail, y_genuine_resume, split_fenetre

toutes_fenetres, tous_masques = [], []
tous_labels_dq_resume, tous_labels_dq_detail = [], []
tous_labels_genuine, tous_splits, tous_foyers = [], [], []

for foyer, groupe in df.groupby('LCLid'):
    groupe = groupe.sort_values('tstp')
    f, m, ydr, ydd, ygr, sp = construire_fenetres(groupe, LONGUEUR_FENETRE, STRIDE)
    toutes_fenetres.extend(f)
    tous_masques.extend(m)
    tous_labels_dq_resume.extend(ydr)
    tous_labels_dq_detail.extend(ydd)
    tous_labels_genuine.extend(ygr)
    tous_splits.extend(sp)
    tous_foyers.extend([foyer] * len(f))

X = np.array(toutes_fenetres)
masques_validite = np.array(tous_masques)
y_dq = np.array(tous_labels_dq_resume)
y_dq_detail = np.array(tous_labels_dq_detail)
y_genuine = np.array(tous_labels_genuine)
splits_array = np.array(tous_splits)

print("Shape tenseur complet :", X.shape)
print("Répartition split :", pd.Series(splits_array).value_counts())
print("\nTaux data-quality par split :")
print(pd.DataFrame({'split': splits_array, 'dq': y_dq}).groupby('split')['dq'].mean())
print("\nTaux genuine_event par split :")
print(pd.DataFrame({'split': splits_array, 'genuine': y_genuine}).groupby('split')['genuine'].mean())

Shape tenseur complet : (39539, 48, 5)
Répartition split : train    32027
test      7471
mixte       41
Name: count, dtype: int64

Taux data-quality par split :
split
mixte    0.146341
test     0.205863
train    0.228588
Name: dq, dtype: float64

Taux genuine_event par split :
split
mixte    0.000000
test     0.016598
train    0.026634
Name: genuine, dtype: float64


Normalisation (fit uniquement sur fenêtres train normales)

In [ ]:
mask_train_normal = (splits_array == 'train') & (~y_dq) & (~y_genuine)
X_train_normal = X[mask_train_normal]

moyenne_canaux = X_train_normal.reshape(-1, len(CANAUX)).mean(axis=0)
ecart_type_canaux = X_train_normal.reshape(-1, len(CANAUX)).std(axis=0)

X_normalise = (X - moyenne_canaux) / ecart_type_canaux

print("Fenêtres utilisées pour la normalisation (train, 100% normales) :", mask_train_normal.sum())
print("Moyennes par canal :", dict(zip(CANAUX, moyenne_canaux.round(3))))
print("Écarts-types par canal :", dict(zip(CANAUX, ecart_type_canaux.round(3))))

Fenêtres utilisées pour la normalisation (train, 100% normales) : 24001
Moyennes par canal : {'energy(kWh/hh)': np.float64(0.433), 'Voltage': np.float64(241.111), 'Global_intensity': np.float64(3.815), 'Temperature_synth': np.float64(18.372), 'Humidity_synth': np.float64(69.155)}
Écarts-types par canal : {'energy(kWh/hh)': np.float64(0.522), 'Voltage': np.float64(4.003), 'Global_intensity': np.float64(4.345), 'Temperature_synth': np.float64(5.379), 'Humidity_synth': np.float64(14.696)}


Sauvegarde finale

In [ ]:
mask_train = splits_array == 'train'
mask_test = splits_array == 'test'

np.savez(base + 'tenseurs_dl.npz',
         X_train=X_normalise[mask_train], y_train_dq=y_dq[mask_train], y_train_genuine=y_genuine[mask_train],
         y_train_dq_detail=y_dq_detail[mask_train],
         X_test=X_normalise[mask_test], y_test_dq=y_dq[mask_test], y_test_genuine=y_genuine[mask_test],
         y_test_dq_detail=y_dq_detail[mask_test],
         moyenne_canaux=moyenne_canaux, ecart_type_canaux=ecart_type_canaux,
         canaux=np.array(CANAUX))

print("Train :", X_normalise[mask_train].shape, "- Test :", X_normalise[mask_test].shape)

Train : (32027, 48, 5) - Test : (7471, 48, 5)
